# Brasileirão - 2006 a 2026
>Análises estatísticas dos campeonatos brasileiros na era dos pontos corridos, com 20 times e 38 rodadas.

# Dataprep

# Import

In [1]:
import time
import random
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# scrapping
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# plot
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import root_mean_squared_error as rmse

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)

# Dataprep


## Load data from 2006 to 2025

In [2]:
df_2006_2024 = pd.read_csv('../dados/brasileirao_2006_2024.csv')
df_2025 = pd.read_csv('../dados/brasileirao_2025.csv')
df_2006_2025 = pd.concat([df_2006_2024, df_2025])
# df_2006_2024.rename(columns = {'time': 'time_mandante',
#                                     'adversário':'time_visitante',
#                                     'gols_pro':'gols_mandante',
#                                     'gols_contra': 'gols_visitante'
#                                     }, inplace = True)
df_2006_2025

,ano_campeonato,rodada,time_mandante,time_visitante,gols_mandante,gols_visitante
0,2006,1,Atlético-PR,Fluminense,1.0,2.0
1,2006,1,Botafogo,Fortaleza,1.0,0.0
2,2006,1,Goiás,Santos,0.0,0.0
3,2006,1,Grêmio,Corinthians,2.0,0.0
4,2006,1,Juventude,Paraná,1.0,0.0
...,...,...,...,...,...,...
375,2025,38,INT,RBB,3.0,1.0
376,2025,38,MIR,FLA,3.0,3.0
377,2025,38,SAN,CRU,3.0,0.0
378,2025,38,SPT,GRE,0.0,4.0


## 2026

In [3]:
# -------------------------
# Config
# -------------------------
URL = "https://ge.globo.com/futebol/brasileirao-serie-a/"
ANO = 2026
PRIMEIRA_RODADA = 1
ULTIMA_RODADA = 38
OUTPUT_CSV = "../dados/brasileirao_2026.csv"


# -------------------------
# Helpers
# -------------------------
def parse_int_safe(s):
    if s is None:
        return None

    s = str(s).strip()

    if not s:
        return None

    m = re.search(r"(\d+)", s)

    return int(m.group(1)) if m else None


def human_pause(a=0.6, b=1.4):
    time.sleep(random.uniform(a, b))


def extrair_texto_por_xpath(elemento, xpaths):
    """
    Tenta extrair texto usando uma lista de XPaths.
    Retorna o primeiro texto válido encontrado.
    """
    for xp in xpaths:
        try:
            txt = elemento.find_element(By.XPATH, xp).text.strip()
            if txt:
                return txt
        except Exception:
            continue

    return None


def extrair_texto_por_css(elemento, seletores):
    """
    Tenta extrair texto usando uma lista de seletores CSS.
    Retorna o primeiro texto válido encontrado.
    """
    for sel in seletores:
        try:
            txt = elemento.find_element(By.CSS_SELECTOR, sel).text.strip()
            if txt:
                return txt
        except Exception:
            continue

    return None


# -------------------------
# Cria driver com stealth manual
# -------------------------
def criar_driver_stealth():
    options = webdriver.ChromeOptions()

    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/129.0.6668.89 Safari/537.36"
    )

    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-infobars")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-gpu")

    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    driver.maximize_window()

    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {
            "source": """
                Object.defineProperty(navigator, 'webdriver', { get: () => undefined });
                Object.defineProperty(navigator, 'languages', { get: () => ['pt-BR','pt','en-US'] });
                Object.defineProperty(navigator, 'plugins', { get: () => [1,2,3,4,5] });
                window.chrome = window.chrome || { runtime: {} };

                (function() {
                    const getParam = WebGLRenderingContext.prototype.getParameter;
                    WebGLRenderingContext.prototype.getParameter = function(param) {
                        if (param === 37445) return 'Intel Inc.';
                        if (param === 37446) return 'Intel Iris OpenGL Engine';
                        return getParam.call(this, param);
                    };
                })();

                try {
                    Object.defineProperty(window, 'Notification', {
                        get: () => ({ permission: 'granted' })
                    });
                } catch(e) {}
            """
        },
    )

    return driver


# -------------------------
# Cookies
# -------------------------
def accept_cookies_if_any(driver, wait):
    selectors = [
        ".cookie-banner-lgpd_accept-button",
        "button.cookie-accept",
        "button[class*='cookie'][class*='accept']",
        "button[data-testid='cookie-accept']",
        "button.cookie-banner__btn",
    ]

    for sel in selectors:
        try:
            btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, sel)))
            human_pause(0.6, 1.2)
            btn.click()
            print("Cookies aceitos pelo seletor:", sel)
            return True
        except Exception:
            continue

    print("Popup de cookies não encontrado ou já aceito.")
    return False


# -------------------------
# Rodada atual
# -------------------------
def get_current_round(driver):
    possible = [
        "span.lista-jogos__navegacao--rodada",
        ".lista-jogos__navegacao--rodada",
        "span.navegacao-rodada__numero",
        ".navegacao-rodada__numero",
    ]

    for sel in possible:
        try:
            el = driver.find_element(By.CSS_SELECTOR, sel)
            text = el.text.strip()
            m = re.search(r"(\d{1,2})", text)
            if m:
                return int(m.group(1))
        except Exception:
            continue

    try:
        body_text = driver.find_element(By.TAG_NAME, "body").text
        m = re.search(r"Rodad[ae]\s*(\d{1,2})", body_text, re.IGNORECASE)
        if m:
            return int(m.group(1))
    except Exception:
        pass

    return None


# -------------------------
# Navegação
# -------------------------
def click_left(driver, wait):
    selectors = [
        ".lista-jogos__navegacao--seta-esquerda",
        ".lista-jogos__navegacao--setas-ativa.lista-jogos__navegacao--seta-esquerda",
        "button[aria-label*='Anterior']",
        "button[aria-label*='anterior']",
        "button[aria-label*='left']",
    ]

    for sel in selectors:
        try:
            el = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, sel)))
            human_pause(0.4, 1.0)
            el.click()
            return True
        except Exception:
            continue

    return False


def click_right(driver, wait):
    selectors = [
        ".lista-jogos__navegacao--seta-direita",
        ".lista-jogos__navegacao--setas-ativa.lista-jogos__navegacao--seta-direita",
        "button[aria-label*='Próxima']",
        "button[aria-label*='proxima']",
        "button[aria-label*='right']",
    ]

    for sel in selectors:
        try:
            el = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, sel)))
            human_pause(0.4, 1.0)
            el.click()
            return True
        except Exception:
            continue

    return False


# -------------------------
# Extração dos jogos
# -------------------------
def extrair_jogos_da_rodada(driver, rodada_num):
    """
    Extrai os jogos da rodada exibida no site.
    A extração dos times foi deixada mais robusta porque o HTML do GE
    pode variar entre nome, sigla, span interno ou atributo.
    """

    dados = []

    try:
        rodada_container = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "lista-jogos"))
        )
    except Exception:
        print(f"❌ Rodada {rodada_num}: container '.lista-jogos' não encontrado.")
        return dados

    jogos = rodada_container.find_elements(By.CLASS_NAME, "lista-jogos__jogo")

    if not jogos:
        print(f"⚠ Rodada {rodada_num}: nenhum elemento '.lista-jogos__jogo' encontrado.")
        return dados

    for idx, jogo in enumerate(jogos, start=1):

        # -------------------------
        # Mandante
        # -------------------------
        xpaths_mandante = [
            ".//*[contains(@class, 'placar__equipes--mandante')]//*[contains(@class, 'equipes__sigla')]",
            ".//*[contains(@class, 'placar__equipes--mandante')]//*[contains(@class, 'equipes__nome-sigla')]",
            ".//*[contains(@class, 'placar__equipes--mandante')]//*[contains(@class, 'equipes__nome')]",
            ".//*[contains(@class, 'placar__equipes--mandante')]//*[contains(@class, 'time')]",
            ".//*[contains(@class, 'placar__equipes--mandante')]//span",
            ".//*[contains(@class, 'placar__equipes--mandante')]//strong",
            ".//*[contains(@class, 'mandante')]//*[contains(@class, 'equipes__sigla')]",
            ".//*[contains(@class, 'mandante')]//*[contains(@class, 'equipes__nome')]",
            ".//*[contains(@class, 'mandante')]//span",
        ]

        css_mandante = [
            ".placar__equipes--mandante .equipes__sigla",
            ".placar__equipes--mandante .equipes__nome-sigla",
            ".placar__equipes--mandante .equipes__nome",
            ".placar__equipes--mandante span",
            ".placar__equipes--mandante strong",
        ]

        mandante = extrair_texto_por_xpath(jogo, xpaths_mandante)

        if mandante is None:
            mandante = extrair_texto_por_css(jogo, css_mandante)

        # -------------------------
        # Visitante
        # -------------------------
        xpaths_visitante = [
            ".//*[contains(@class, 'placar__equipes--visitante')]//*[contains(@class, 'equipes__sigla')]",
            ".//*[contains(@class, 'placar__equipes--visitante')]//*[contains(@class, 'equipes__nome-sigla')]",
            ".//*[contains(@class, 'placar__equipes--visitante')]//*[contains(@class, 'equipes__nome')]",
            ".//*[contains(@class, 'placar__equipes--visitante')]//*[contains(@class, 'time')]",
            ".//*[contains(@class, 'placar__equipes--visitante')]//span",
            ".//*[contains(@class, 'placar__equipes--visitante')]//strong",
            ".//*[contains(@class, 'visitante')]//*[contains(@class, 'equipes__sigla')]",
            ".//*[contains(@class, 'visitante')]//*[contains(@class, 'equipes__nome')]",
            ".//*[contains(@class, 'visitante')]//span",
        ]

        css_visitante = [
            ".placar__equipes--visitante .equipes__sigla",
            ".placar__equipes--visitante .equipes__nome-sigla",
            ".placar__equipes--visitante .equipes__nome",
            ".placar__equipes--visitante span",
            ".placar__equipes--visitante strong",
        ]

        visitante = extrair_texto_por_xpath(jogo, xpaths_visitante)

        if visitante is None:
            visitante = extrair_texto_por_css(jogo, css_visitante)

        # -------------------------
        # Fallback por texto completo do jogo
        # -------------------------
        if mandante is None or visitante is None:
            texto_jogo = jogo.text.strip()
            linhas = [linha.strip() for linha in texto_jogo.split("\n") if linha.strip()]

            # Tenta capturar siglas/nomes em linhas de texto.
            # Normalmente o bloco contém algo como:
            # TIME A
            # 1
            # x
            # 0
            # TIME B
            candidatos = [
                linha for linha in linhas
                if not re.fullmatch(r"\d+", linha)
                and linha.lower() not in ["x", "pré-jogo", "encerrado", "adiado"]
                and not re.search(r"\d{1,2}:\d{2}", linha)
            ]

            if mandante is None and len(candidatos) >= 1:
                mandante = candidatos[0]

            if visitante is None and len(candidatos) >= 2:
                visitante = candidatos[-1]

        # -------------------------
        # Gols mandante
        # -------------------------
        try:
            gols_mandante = jogo.find_element(
                By.CSS_SELECTOR,
                ".placar-box__valor.placar-box__valor--mandante"
            ).text.strip()
        except Exception:
            gols_mandante = None

        # -------------------------
        # Gols visitante
        # -------------------------
        try:
            gols_visitante = jogo.find_element(
                By.CSS_SELECTOR,
                ".placar-box__valor.placar-box__valor--visitante"
            ).text.strip()
        except Exception:
            gols_visitante = None

        # -------------------------
        # Debug se não encontrar times
        # -------------------------
        if mandante is None or visitante is None:
            print(f"⚠ Rodada {rodada_num}, jogo {idx}: não foi possível capturar mandante/visitante.")
            print("Texto do bloco:")
            print(jogo.text[:500])
            print("HTML parcial:")
            print(jogo.get_attribute("outerHTML")[:1000])

        dados.append({
            "ano_campeonato": ANO,
            "rodada": rodada_num,
            "time_mandante": mandante,
            "time_visitante": visitante,
            "gols_mandante": gols_mandante,
            "gols_visitante": gols_visitante,
        })

    print(f"✓ Rodada {rodada_num}: {len(dados)} jogos extraídos")

    return dados


# -------------------------
# Fluxo principal
# -------------------------
def main():
    driver = criar_driver_stealth()
    wait = WebDriverWait(driver, 15)

    driver.get(URL)

    time.sleep(2 + random.random())

    accept_cookies_if_any(driver, wait)

    # Detecta a rodada máxima disponível no site.
    rodada_atual = get_current_round(driver)

    if rodada_atual is None:
        print("Não foi possível detectar rodada inicial; assumindo ULTIMA_RODADA.")
        rodada_atual = ULTIMA_RODADA

    rodada_max_disponivel = rodada_atual

    print("Rodada máxima disponível detectada:", rodada_max_disponivel)

    # Volta até a primeira rodada.
    attempts = 0

    while rodada_atual is not None and rodada_atual > PRIMEIRA_RODADA and attempts < 60:
        ok = click_left(driver, wait)

        human_pause(0.8, 1.6)

        nova = get_current_round(driver)

        if nova:
            rodada_atual = nova

        attempts += 1

        print(f"Voltando... agora na rodada {rodada_atual} (tent {attempts})")

        if not ok:
            try:
                driver.execute_script("window.scrollBy(0, 200);")
                human_pause(0.4, 0.8)
            except Exception:
                pass

    rodada_detalhe = get_current_round(driver) or rodada_atual
    print("Rodada onde iniciaremos extração:", rodada_detalhe)

    all_rows = []

    # Extrai somente da rodada 1 até a rodada máxima disponível.
    for target_round in range(PRIMEIRA_RODADA, rodada_max_disponivel + 1):

        shown = get_current_round(driver)

        if shown is None:
            shown = target_round

        print(f"\nExtraindo rodada exibida {shown} (alvo {target_round})")

        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CLASS_NAME, "lista-jogos"))
            )
        except Exception:
            print("Atenção: container '.lista-jogos' não carregou rapidamente.")

        human_pause(0.8, 1.4)

        rows = extrair_jogos_da_rodada(driver, shown)

        if not rows:
            print(f"Aviso: nenhum jogo extraído na rodada {shown}.")

        all_rows.extend(rows)

        if shown >= rodada_max_disponivel:
            break

        ok = click_right(driver, wait)

        if not ok:
            print("Falha ao clicar seta direita. Tentando scroll e novo clique.")
            try:
                driver.execute_script("window.scrollBy(0, 200);")
                human_pause(0.5, 1.0)
                click_right(driver, wait)
            except Exception:
                print("Não foi possível avançar para próxima rodada por falta de seletor.")

        human_pause(0.8, 1.6)

    try:
        driver.quit()
    except Exception:
        pass

    df_actual_year = pd.DataFrame(
        all_rows,
        columns=[
            "ano_campeonato",
            "rodada",
            "time_mandante",
            "time_visitante",
            "gols_mandante",
            "gols_visitante",
        ],
    )

    if not df_actual_year.empty:
        df_actual_year = (
            df_actual_year
            .sort_values(by=["rodada", "time_mandante"], na_position="last")
            .reset_index(drop=True)
        )

    df_actual_year.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"\nConcluído. Arquivo salvo: {OUTPUT_CSV}")
    print(f"Rodadas extraídas: {PRIMEIRA_RODADA} até {rodada_max_disponivel}")
    print(df_actual_year.tail())

    return df_actual_year


if __name__ == "__main__":
    main()

Cookies aceitos pelo seletor: .cookie-banner-lgpd_accept-button
Rodada máxima disponível detectada: 15
Voltando... agora na rodada 14 (tent 1)
Voltando... agora na rodada 13 (tent 2)
Voltando... agora na rodada 12 (tent 3)
Voltando... agora na rodada 11 (tent 4)
Voltando... agora na rodada 10 (tent 5)
Voltando... agora na rodada 9 (tent 6)
Voltando... agora na rodada 8 (tent 7)
Voltando... agora na rodada 7 (tent 8)
Voltando... agora na rodada 6 (tent 9)
Voltando... agora na rodada 5 (tent 10)
Voltando... agora na rodada 4 (tent 11)
Voltando... agora na rodada 3 (tent 12)
Voltando... agora na rodada 2 (tent 13)
Voltando... agora na rodada 1 (tent 14)
Rodada onde iniciaremos extração: 1

Extraindo rodada exibida 1 (alvo 1)
✓ Rodada 1: 10 jogos extraídos

Extraindo rodada exibida 2 (alvo 2)
✓ Rodada 2: 10 jogos extraídos

Extraindo rodada exibida 3 (alvo 3)
✓ Rodada 3: 10 jogos extraídos

Extraindo rodada exibida 4 (alvo 4)
✓ Rodada 4: 10 jogos extraídos

Extraindo rodada exibida 5 (alvo

# Concatenando os df's

In [4]:
df_2026 = pd.read_csv('../dados/brasileirao_2026.csv')

df = pd.concat([df_2006_2025, df_2026])
df.sort_values(['ano_campeonato','rodada','time_mandante'], inplace = True)
try:
    df['time_mandante'] = df['time_mandante'].apply(lambda x: x.strip())
except:
    pass
try:
    df['time_visitante'] = df['time_visitante'].apply(lambda x: x.strip())
except:
    pass

df = df.dropna(subset = ['gols_mandante', 'gols_visitante', 'time_mandante', 'time_visitante'])
print(f'Antes do drop_duplicates: {df.shape}')
df.drop_duplicates(subset = ['ano_campeonato', 'rodada', 'time_mandante', 'time_visitante'], keep = 'last', inplace=True)
print(f'Após o drop_duplicates: {df.shape}')
df

Antes do drop_duplicates: (7746, 6)
Após o drop_duplicates: (7746, 6)


,ano_campeonato,rodada,time_mandante,time_visitante,gols_mandante,gols_visitante
0,2006,1,Atlético-PR,Fluminense,1.0,2.0
1,2006,1,Botafogo,Fortaleza,1.0,0.0
2,2006,1,Goiás,Santos,0.0,0.0
3,2006,1,Grêmio,Corinthians,2.0,0.0
4,2006,1,Juventude,Paraná,1.0,0.0
...,...,...,...,...,...,...
145,2026,15,GRE,FLA,0.0,1.0
146,2026,15,MIR,CHA,1.0,1.0
147,2026,15,REM,PAL,1.0,1.0
148,2026,15,SAN,RBB,2.0,0.0


# Unificando nomes de times similares

In [5]:
dict_times_unificacao = {'Athletico-PR' : 'Atlético-PR',
                         'CAP' : 'Atlético-PR',
                         'Goiás EC' : 'Goiás',
                         'Santos FC' : 'Santos',
                         'BAH':'Bahia',
                         'EC Bahia':'Bahia',
                         'CRU':'Cruzeiro',
                         'FLA':'Flamengo',
                         'FOR':'Fortaleza',
                         'GRE':'Grêmio',
                         'JUV':'Juventude',
                         'PAL':'Palmeiras',
                         'RBB':'Bragantino',
                         'RB Bragantino':'Bragantino',
                         'SAO':'São Paulo',
                         'VAS':'Vasco',
                         'Vasco da Gama':'Vasco',
                         'BOT':'Botafogo',
                         'CAM':'Atlético-MG',
                         'CEA':'Ceará SC',
                         'COR':'Corinthians',
                         'FLU':'Fluminense', 
                         'INT':'Internacional', 
                         'MIR':'Mirassol', 
                         'SAN':'Santos',
                         'SPT':'Sport Recife', 
                         'VIT':'Vitória',
                         'EC Vitória':'Vitória',
                         'REM': 'Remo',
                         'CFC': 'Coritiba',
                         'Coritiba FC': 'Coritiba',
                         'CHA': 'Chapecoense',
                         }

df['time_mandante'].replace(dict_times_unificacao, inplace = True)
df['time_visitante'].replace(dict_times_unificacao, inplace = True)

display(np.sort(df['time_mandante'].unique()))
display(np.sort(df['time_visitante'].unique()))

C:\Users\M4005001\AppData\Local\Temp\ipykernel_17740\4013568732.py:35: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['time_mandante'].replace(dict_times_unificacao, inplace = True)
C:\Users\M4005001\AppData\Local\Temp\ipykernel_17740\4013568732.py:36: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always b

array(['América-MG', 'América-RN', 'Atlético-GO', 'Atlético-MG',
       'Atlético-PR', 'Avaí FC', 'Bahia', 'Barueri', 'Botafogo',
       'Bragantino', 'CSA', 'Ceará SC', 'Chapecoense', 'Corinthians',
       'Coritiba', 'Criciúma EC', 'Cruzeiro', 'Cuiabá-MT',
       'Figueirense FC', 'Flamengo', 'Fluminense', 'Fortaleza', 'Goiás',
       'Grêmio', 'Guarani', 'Internacional', 'Ipatinga FC',
       'Joinville-SC', 'Juventude', 'Mirassol', 'Náutico', 'Palmeiras',
       'Paraná', 'Ponte Preta', 'Portuguesa', 'Remo', 'Santa Cruz',
       'Santo André', 'Santos', 'Sport Recife', 'São Caetano',
       'São Paulo', 'Vasco', 'Vitória'], dtype=object)

array(['América-MG', 'América-RN', 'Atlético-GO', 'Atlético-MG',
       'Atlético-PR', 'Avaí FC', 'Bahia', 'Barueri', 'Botafogo',
       'Bragantino', 'CSA', 'Ceará SC', 'Chapecoense', 'Corinthians',
       'Coritiba', 'Criciúma EC', 'Cruzeiro', 'Cuiabá-MT',
       'Figueirense FC', 'Flamengo', 'Fluminense', 'Fortaleza', 'Goiás',
       'Grêmio', 'Guarani', 'Internacional', 'Ipatinga FC',
       'Joinville-SC', 'Juventude', 'Mirassol', 'Náutico', 'Palmeiras',
       'Paraná', 'Ponte Preta', 'Portuguesa', 'Remo', 'Santa Cruz',
       'Santo André', 'Santos', 'Sport Recife', 'São Caetano',
       'São Paulo', 'Vasco', 'Vitória'], dtype=object)

# Estatística básica

In [6]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
ano_campeonato,7746.0,2015.699200,5.889001,2006.0,2011.0,2016.0,2021.0,2026.0
rodada,7746.0,19.280919,10.988008,1.0,10.0,19.0,29.0,38.0
gols_mandante,7746.0,1.488639,1.193781,0.0,1.0,1.0,2.0,8.0
gols_visitante,7746.0,1.001033,1.005215,0.0,0.0,1.0,2.0,7.0


# Testes de consistência dos dados - Parte 1

In [7]:
#rodadas por temporada: esperadas 38 (#ok)
display(df.groupby('ano_campeonato')['rodada'].max())

#jogos por campeonato: esperados 380 (#ok)
display(df.groupby(['ano_campeonato'])['time_mandante'].count())



ano_campeonato
2006    38
2007    38
2008    38
2009    38
2010    38
2011    38
2012    38
2013    38
2014    38
2015    38
2016    38
2017    38
2018    38
2019    38
2020    38
2021    38
2022    38
2023    38
2024    38
2025    38
2026    15
Name: rodada, dtype: int64

ano_campeonato
2006    380
2007    380
2008    380
2009    380
2010    380
2011    380
2012    380
2013    380
2014    380
2015    380
2016    379
2017    380
2018    380
2019    380
2020    380
2021    380
2022    380
2023    380
2024    380
2025    380
2026    147
Name: time_mandante, dtype: int64

In [8]:
#jogos como mandante por time por temporada: esperados 19 jogos ao final do campeonato
df_jogos_por_time_mandante = df.groupby(['ano_campeonato','time_mandante']).count()#.reset_index()
display('Jogos por time como mandantes, diferente de 19 jogos:',df_jogos_por_time_mandante.loc[df_jogos_por_time_mandante.rodada!=19].iloc[:,:1])#.sort_values('time_mandante'))

#jogos como visitante por time por temporada: esperados 19 jogos ao final do campeonato
df_jogos_por_time_visitante = df.groupby(['ano_campeonato','time_visitante']).count()#.reset_index()
display('Jogos por time como visitante, diferente de 19 jogos:',df_jogos_por_time_visitante.loc[df_jogos_por_time_visitante.rodada!=19].iloc[:,:1])#.sort_values('time_visitante'))

# Todos os times apresentaram 19 jogos como mandantes e 19 como visitantes por ano, totalizando 38 jogos por campeonato

'Jogos por time como mandantes, diferente de 19 jogos:'

rodada
ano_campeonato time_mandante        
2016           Chapecoense        18
2026           Atlético-MG         7
               Atlético-PR         8
               Bahia               7
               Botafogo            6
               Bragantino          7
               Chapecoense         8
               Corinthians         8
               Coritiba            7
               Cruzeiro            8
               Flamengo            6
               Fluminense          8
               Grêmio              7
               Internacional       8
               Mirassol            8
               Palmeiras           7
               Remo                7
               Santos              8
               São Paulo           7
               Vasco               8
               Vitória             7

'Jogos por time como visitante, diferente de 19 jogos:'

rodada
ano_campeonato time_visitante        
2016           Atlético-MG         18
2026           Atlético-MG          8
               Atlético-PR          7
               Bahia                7
               Botafogo             8
               Bragantino           8
               Chapecoense          6
               Corinthians          7
               Coritiba             8
               Cruzeiro             7
               Flamengo             8
               Fluminense           7
               Grêmio               8
               Internacional        7
               Mirassol             6
               Palmeiras            8
               Remo                 8
               Santos               7
               São Paulo            8
               Vasco                7
               Vitória              7

# Engenharia de features

In [9]:
df['pontos_mandante'] = df.gols_mandante - df.gols_visitante
df['pontos_mandante'] = df['pontos_mandante'].map(lambda x: 3 if x > 0 else 1 if x == 0 else 0)

df['pontos_visitante'] = df.gols_visitante - df.gols_mandante
df['pontos_visitante'] = df['pontos_visitante'].map(lambda x: 3 if x > 0 else 1 if x == 0 else 0)

df

,ano_campeonato,rodada,time_mandante,time_visitante,gols_mandante,gols_visitante,pontos_mandante,pontos_visitante
0,2006,1,Atlético-PR,Fluminense,1.0,2.0,0,3
1,2006,1,Botafogo,Fortaleza,1.0,0.0,3,0
2,2006,1,Goiás,Santos,0.0,0.0,1,1
3,2006,1,Grêmio,Corinthians,2.0,0.0,3,0
4,2006,1,Juventude,Paraná,1.0,0.0,3,0
...,...,...,...,...,...,...,...,...
145,2026,15,Grêmio,Flamengo,0.0,1.0,0,3
146,2026,15,Mirassol,Chapecoense,1.0,1.0,1,1
147,2026,15,Remo,Palmeiras,1.0,1.0,1,1
148,2026,15,Santos,Bragantino,2.0,0.0,3,0


In [10]:
#criação de df_mandante e df_visitante para depois concatená-los

#df_mandante
df_mandante = df[['ano_campeonato',	'rodada',	'time_mandante','time_visitante','gols_mandante','gols_visitante']].copy()
df_mandante.rename(columns = {'time_mandante':'time',
                              'time_visitante':'adversário',
                              'gols_mandante':'gols_pro',
                              'gols_visitante': 'gols_contra'}, inplace = True)
df_mandante['pontos'] = df_mandante.gols_pro - df_mandante.gols_contra
df_mandante['pontos'] = df_mandante['pontos'].map(lambda x: 3 if x > 0 else 1 if x == 0 else 0)

#df_visitante
df_visitante = df[['ano_campeonato',	'rodada',	'time_visitante','time_mandante','gols_mandante','gols_visitante']].copy()
df_visitante.rename(columns = {'time_visitante':'time',
                               'time_mandante': 'adversário',
                              'gols_visitante':'gols_pro',
                              'gols_mandante': 'gols_contra'
                              }, inplace = True)

df_visitante['pontos'] = df_visitante.gols_pro - df_visitante.gols_contra
df_visitante['pontos'] = df_visitante['pontos'].map(lambda x: 3 if x > 0 else 1 if x == 0 else 0)

df_concat = pd.concat([df_mandante, df_visitante]).reset_index(drop = True)

#criação das features vitoria, empate, derrota
df_concat['vitoria'] = df_concat['pontos'].map(lambda x: 1 if x == 3 else 0)
df_concat['derrota'] = df_concat['pontos'].map(lambda x: 1 if x == 0 else 0)
df_concat['empate'] = df_concat['pontos'].map(lambda x: 1 if x == 1 else 0)

#anulando dados de rodadas que ainda não aconteceram
df_concat.loc[df_concat.gols_pro != df_concat.gols_pro, ['pontos', 'vitoria', 'derrota', 'empate']] = np.nan

df_concat

,ano_campeonato,rodada,time,adversário,gols_pro,gols_contra,pontos,vitoria,derrota,empate
0,2006,1,Atlético-PR,Fluminense,1.0,2.0,0.0,0.0,1.0,0.0
1,2006,1,Botafogo,Fortaleza,1.0,0.0,3.0,1.0,0.0,0.0
2,2006,1,Goiás,Santos,0.0,0.0,1.0,0.0,0.0,1.0
3,2006,1,Grêmio,Corinthians,2.0,0.0,3.0,1.0,0.0,0.0
4,2006,1,Juventude,Paraná,1.0,0.0,3.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
15487,2026,15,Flamengo,Grêmio,1.0,0.0,3.0,1.0,0.0,0.0
15488,2026,15,Chapecoense,Mirassol,1.0,1.0,1.0,0.0,0.0,1.0
15489,2026,15,Palmeiras,Remo,1.0,1.0,1.0,0.0,0.0,1.0
15490,2026,15,Bragantino,Santos,0.0,2.0,0.0,0.0,1.0,0.0


# Testes de consistência dos dados - Parte 2

In [11]:
#jogos por temporada: esperados 38 por time (OK)
display(df_concat.groupby(['time','ano_campeonato'])['rodada'].count())
display(df_concat.groupby(['time','ano_campeonato'])['rodada'].count().describe())


#times por temporada: esperados 20
display(df_concat.groupby('ano_campeonato')['time'].nunique())

time            ano_campeonato
América-MG      2011              38
                2016              38
                2018              38
                2021              38
                2022              38
                2023              38
América-RN      2007              38
Atlético-GO     2010              38
                2011              38
                2012              38
                2017              38
                2020              38
                2021              38
                2022              38
                2024              38
Atlético-MG     2007              38
                2008              38
                2009              38
                2010              38
                2011              38
                2012              38
                2013              38
                2014              38
                2015              38
                2016              37
                2017              38
       

count    420.000000
mean      36.885714
std        4.968280
min       14.000000
25%       38.000000
50%       38.000000
75%       38.000000
max       38.000000
Name: rodada, dtype: float64

ano_campeonato
2006    20
2007    20
2008    20
2009    20
2010    20
2011    20
2012    20
2013    20
2014    20
2015    20
2016    20
2017    20
2018    20
2019    20
2020    20
2021    20
2022    20
2023    20
2024    20
2025    20
2026    20
Name: time, dtype: int64

# Identificando os campeões de cada ano | 1o turno e final

## Ajuste da base para a rodada#38 de 2016, jogo entre Chapecoense e Atletico MG
* Em função do acidente ocorrido com o avião com o time da Chapecoense em 2016, o jogo da rodada #38 teve WO duplo, ou seja, nenhum dos times compareceram a campo.
* Por tratar-se de uma condição não prevista no regulamento CBF, foi considerada derrota por 3 a 0 para ambos os times.
* Detalhes na matéria: https://www.uol.com.br/esporte/futebol/ultimas-noticias/2016/12/05/presidente-da-chape-diz-que-cbf-ja-cancelou-jogo-contra-o-atletico-mg.htm

* Desta forma a base será ajustada considerando o placar de 0 a 3 para ambos os times.

In [12]:
display('Antes da correção',df_concat.loc[(df_concat.ano_campeonato == 2016) & (df_concat.rodada == 38) & (df_concat.gols_pro.isna())])

df_concat.loc[(df_concat.ano_campeonato == 2016) & (df_concat.rodada == 38) & (df_concat.gols_pro != df_concat.gols_pro), 'gols_contra'] = 3
df_concat.loc[(df_concat.ano_campeonato == 2016) & (df_concat.rodada == 38) & (df_concat.gols_pro != df_concat.gols_pro), 'derrota'] = 1
df_concat.loc[(df_concat.ano_campeonato == 2016) & (df_concat.rodada == 38) & (df_concat.gols_pro != df_concat.gols_pro), ['gols_pro','pontos', 'vitoria','empate']] = 0

display('Após a correção',
        df_concat.loc[(
            (df_concat.rodada == 38) & (df_concat.ano_campeonato == 2016) & ((df_concat.time == 'Chapecoense') | (df_concat.time == 'Atlético-MG')))])

'Antes da correção'

,ano_campeonato,rodada,time,adversário,gols_pro,gols_contra,pontos,vitoria,derrota,empate


'Após a correção'

,ano_campeonato,rodada,time,adversário,gols_pro,gols_contra,pontos,vitoria,derrota,empate


In [13]:
df_cumsum = df_concat.sort_values(['ano_campeonato','time','rodada']).copy()
df_cumsum['pontos_acum'] = df_cumsum.groupby(['time','ano_campeonato'])['pontos'].cumsum()
df_cumsum['jogos_acum'] = df_cumsum['vitoria'] + df_cumsum['derrota'] + df_cumsum['empate']
df_cumsum['jogos_acum'] = df_cumsum.groupby(['time','ano_campeonato'])['jogos_acum'].cumsum()
df_cumsum['vitorias_acum'] = df_cumsum.groupby(['time','ano_campeonato'])['vitoria'].cumsum()
df_cumsum['empates_acum'] = df_cumsum.groupby(['time','ano_campeonato'])['empate'].cumsum()
df_cumsum['derrotas_acum'] = df_cumsum.groupby(['time','ano_campeonato'])['derrota'].cumsum()
df_cumsum['gols_pro_acum'] = df_cumsum.groupby(['time','ano_campeonato'])['gols_pro'].cumsum()
df_cumsum['gols_contra_acum'] = df_cumsum.groupby(['time','ano_campeonato'])['gols_contra'].cumsum()
df_cumsum['saldo_gols_acum'] = df_cumsum['gols_pro_acum'] - df_cumsum['gols_contra_acum']

#ajuste para rodadas que ainda não aconteceram
df_cumsum.loc[df_cumsum.gols_pro != df_cumsum.gols_pro, df_cumsum.columns[6:]] = np.nan

df_cumsum

,ano_campeonato,rodada,time,adversário,gols_pro,gols_contra,pontos,vitoria,derrota,empate,pontos_acum,jogos_acum,vitorias_acum,empates_acum,derrotas_acum,gols_pro_acum,gols_contra_acum,saldo_gols_acum
0,2006,1,Atlético-PR,Fluminense,1.0,2.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,2.0,-1.0
7765,2006,2,Atlético-PR,Santos,0.0,2.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,2.0,1.0,4.0,-3.0
7766,2006,3,Atlético-PR,Botafogo,4.0,0.0,3.0,1.0,0.0,0.0,3.0,3.0,1.0,0.0,2.0,5.0,4.0,1.0
30,2006,4,Atlético-PR,Internacional,1.0,2.0,0.0,0.0,1.0,0.0,3.0,4.0,1.0,0.0,3.0,6.0,6.0,0.0
7793,2006,5,Atlético-PR,Santa Cruz,2.0,1.0,3.0,1.0,0.0,0.0,6.0,5.0,2.0,0.0,3.0,8.0,7.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7705,2026,11,Vitória,São Paulo,2.0,0.0,3.0,1.0,0.0,0.0,14.0,10.0,4.0,2.0,4.0,11.0,14.0,-3.0
7715,2026,12,Vitória,Corinthians,0.0,0.0,1.0,0.0,0.0,1.0,15.0,11.0,4.0,3.0,4.0,11.0,14.0,-3.0
15465,2026,13,Vitória,Atlético-PR,1.0,3.0,0.0,0.0,1.0,0.0,15.0,12.0,4.0,3.0,5.0,12.0,17.0,-5.0
7735,2026,14,Vitória,Coritiba,4.0,1.0,3.0,1.0,0.0,0.0,18.0,13.0,5.0,3.0,5.0,16.0,18.0,-2.0


In [14]:
#verificação se todas as rodadas entre 2006 e 2026 apresentam 20 times distintos
df_check = df_cumsum.groupby(['ano_campeonato','rodada']).count().reset_index()[['ano_campeonato','rodada','time']]
df_check.loc[df_check.time != 20]

,ano_campeonato,rodada,time
417,2016,38,18
763,2026,4,14


In [15]:
# utilizada a feature "gols_pro" pois mesmo que o jogo esteja cadastrado, se ainda não foi realizado, o gols_pro estará como Nan.
# utilizado o denominador 20 pois apesar de cada rodada ter 10 jogos, cada linha refere-se aos jogos de cada time, logo, mandante e visitante terão linhas exclusivas, por isso 20 linhas por rodada
# a divisao de inteiros por 20 resulta em quantas rodadas completas teremos no df
df_cumsum.dropna(subset='gols_pro').groupby('ano_campeonato')['gols_pro'].count() // 20 

ano_campeonato
2006    38
2007    38
2008    38
2009    38
2010    38
2011    38
2012    38
2013    38
2014    38
2015    38
2016    37
2017    38
2018    38
2019    38
2020    38
2021    38
2022    38
2023    38
2024    38
2025    38
2026    14
Name: gols_pro, dtype: int64

In [16]:
# Critérios de desempate do Brasileirão:
# 1º pontos
# 2º vitórias
# 3º saldo de gols
# 4º gols pró

criterios = [
    'ano_campeonato',
    'pontos_acum',
    'vitorias_acum',
    'saldo_gols_acum',
    'gols_pro_acum'
]

# Garantia contra duplicidades por time/rodada
df_cumsum = df_cumsum.drop_duplicates(
    subset=['ano_campeonato', 'rodada', 'time'],
    keep='last'
).copy()

# Considera apenas rodadas com jogos realizados
df_realizados = df_cumsum.dropna(subset=['gols_pro']).copy()

# Identifica rodadas completas por ano
df_rodadas_completas = (
    df_realizados
    .groupby(['ano_campeonato', 'rodada'])['time']
    .nunique()
    .reset_index(name='n_times')
)

df_rodadas_completas = df_rodadas_completas.loc[
    df_rodadas_completas['n_times'] == 20
]

# Anos com rodada 19 completa
list_anos_19 = (
    df_rodadas_completas
    .loc[df_rodadas_completas['rodada'] == 19, 'ano_campeonato']
    .unique()
    .tolist()
)

# Anos com rodada 38 completa
list_anos_38 = (
    df_rodadas_completas
    .loc[df_rodadas_completas['rodada'] == 38, 'ano_campeonato']
    .unique()
    .tolist()
)

# -----------------------------
# Classificação do 1º turno
# -----------------------------
df_rodada19 = df_cumsum.loc[
    (df_cumsum['ano_campeonato'].isin(list_anos_19)) &
    (df_cumsum['rodada'] == 19)
].copy()

df_rodada19 = df_rodada19.sort_values(
    ['ano_campeonato', 'pontos_acum', 'vitorias_acum', 'saldo_gols_acum', 'gols_pro_acum'],
    ascending=[True, False, False, False, False]
)

df_rodada19['classificacao_1o_turno'] = (
    df_rodada19
    .groupby('ano_campeonato')
    .cumcount() + 1
)

df_classificacao19_ano = df_rodada19[
    ['ano_campeonato', 'time', 'classificacao_1o_turno']
].copy()

# -----------------------------
# Classificação final
# -----------------------------
df_rodada38 = df_cumsum.loc[
    (df_cumsum['ano_campeonato'].isin(list_anos_38)) &
    (df_cumsum['rodada'] == 38)
].copy()

df_rodada38 = df_rodada38.sort_values(
    ['ano_campeonato', 'pontos_acum', 'vitorias_acum', 'saldo_gols_acum', 'gols_pro_acum'],
    ascending=[True, False, False, False, False]
)

df_rodada38['classificacao_final'] = (
    df_rodada38
    .groupby('ano_campeonato')
    .cumcount() + 1
)

df_classificacao38_ano = df_rodada38[
    ['ano_campeonato', 'time', 'classificacao_final']
].copy()

# -----------------------------
# Merge final
# -----------------------------
df_completo = df_cumsum.merge(
    df_classificacao19_ano,
    on=['ano_campeonato', 'time'],
    how='left'
)

df_completo = df_completo.merge(
    df_classificacao38_ano,
    on=['ano_campeonato', 'time'],
    how='left'
)

df_completo

,ano_campeonato,rodada,time,adversário,gols_pro,gols_contra,pontos,vitoria,derrota,empate,pontos_acum,jogos_acum,vitorias_acum,empates_acum,derrotas_acum,gols_pro_acum,gols_contra_acum,saldo_gols_acum,classificacao_1o_turno,classificacao_final
0,2006,1,Atlético-PR,Fluminense,1.0,2.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,2.0,-1.0,14.0,13.0
1,2006,2,Atlético-PR,Santos,0.0,2.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,2.0,1.0,4.0,-3.0,14.0,13.0
2,2006,3,Atlético-PR,Botafogo,4.0,0.0,3.0,1.0,0.0,0.0,3.0,3.0,1.0,0.0,2.0,5.0,4.0,1.0,14.0,13.0
3,2006,4,Atlético-PR,Internacional,1.0,2.0,0.0,0.0,1.0,0.0,3.0,4.0,1.0,0.0,3.0,6.0,6.0,0.0,14.0,13.0
4,2006,5,Atlético-PR,Santa Cruz,2.0,1.0,3.0,1.0,0.0,0.0,6.0,5.0,2.0,0.0,3.0,8.0,7.0,1.0,14.0,13.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15487,2026,11,Vitória,São Paulo,2.0,0.0,3.0,1.0,0.0,0.0,14.0,10.0,4.0,2.0,4.0,11.0,14.0,-3.0,NaN,NaN
15488,2026,12,Vitória,Corinthians,0.0,0.0,1.0,0.0,0.0,1.0,15.0,11.0,4.0,3.0,4.0,11.0,14.0,-3.0,NaN,NaN
15489,2026,13,Vitória,Atlético-PR,1.0,3.0,0.0,0.0,1.0,0.0,15.0,12.0,4.0,3.0,5.0,12.0,17.0,-5.0,NaN,NaN
15490,2026,14,Vitória,Coritiba,4.0,1.0,3.0,1.0,0.0,0.0,18.0,13.0,5.0,3.0,5.0,16.0,18.0,-2.0,NaN,NaN


# Export to parquet

In [17]:
df_completo.to_parquet('../dados/df_completo.parquet')